# GristMill — Routing Decision Analysis

This notebook analyses routing decisions from the GristMill feedback logs to
identify cost-saving opportunities and monitor routing quality over time.

**Prerequisites:** `pip install -e gristmill-ml matplotlib`

In [ ]:
import json
import sys
from collections import defaultdict
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path('..') / 'src'))

FEEDBACK_DIR = Path('~/.gristmill/feedback').expanduser()

print(f'Feedback dir: {FEEDBACK_DIR}')
print(f'Exists: {FEEDBACK_DIR.exists()}')

In [ ]:
from gristmill_ml.datasets.feedback import FeedbackDataset, ROUTE_LABEL_MAP

# Load both splits so we have all records
ds_train = FeedbackDataset(FEEDBACK_DIR, split='train', min_records=200)
ds_val   = FeedbackDataset(FEEDBACK_DIR, split='val',   min_records=200)
all_records = ds_train.records + ds_val.records

print(f'Total records: {len(all_records)}')
print(f'Date range of timestamps:')
ts_list = sorted(r.timestamp_ms for r in all_records)
if ts_list:
    t0 = datetime.fromtimestamp(ts_list[0] / 1000, tz=timezone.utc)
    t1 = datetime.fromtimestamp(ts_list[-1] / 1000, tz=timezone.utc)
    print(f'  From: {t0.isoformat()}')
    print(f'  To:   {t1.isoformat()}')

In [ ]:
from collections import Counter

decision_counts = Counter(r.route_decision for r in all_records)
labels = list(decision_counts.keys())
sizes  = list(decision_counts.values())

print('Routing decision distribution:')
total = sum(sizes)
for label, count in sorted(decision_counts.items(), key=lambda x: -x[1]):
    pct = count / total * 100
    bar = '#' * int(pct / 2)
    print(f'  {label:<15} {count:>5} ({pct:>5.1f}%)  {bar}')

try:
    import matplotlib.pyplot as plt

    colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors[:len(labels)], startangle=90)
    ax.set_title('Routing Decision Distribution', fontsize=14)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('(matplotlib not installed — skipping pie chart)')

In [ ]:
from collections import defaultdict

conf_by_class = defaultdict(list)
for r in all_records:
    conf_by_class[r.route_decision].append(r.confidence)

print('Confidence statistics per class:')
for cls in sorted(conf_by_class):
    arr = np.array(conf_by_class[cls])
    print(f'  {cls:<15} mean={arr.mean():.3f}  std={arr.std():.3f}  min={arr.min():.3f}  max={arr.max():.3f}')

try:
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, len(conf_by_class), figsize=(14, 4), sharey=True)
    colors = {'LOCAL_ML': '#4CAF50', 'RULES': '#2196F3', 'HYBRID': '#FF9800', 'LLM_NEEDED': '#F44336'}
    for ax, (cls, vals) in zip(axes, sorted(conf_by_class.items())):
        ax.hist(vals, bins=20, color=colors.get(cls, 'gray'), edgecolor='white', alpha=0.8)
        ax.set_title(cls, fontsize=10)
        ax.set_xlabel('Confidence')
    axes[0].set_ylabel('Count')
    fig.suptitle('Confidence Score Distribution per Routing Class', fontsize=12)
    plt.tight_layout()
    plt.show()
except ImportError:
    print('(matplotlib not installed — skipping histogram)')

In [ ]:
# Records where could_have_been_local is True represent potential cost savings
local_possible = [r for r in all_records if r.could_have_been_local is True]

print(f'Records where could_have_been_local=True: {len(local_possible)}')
print(f'  ({len(local_possible) / max(len(all_records), 1) * 100:.1f}% of all records)\n')

if local_possible:
    print('Sample records (first 5):')
    for r in local_possible[:5]:
        text = getattr(r, '_text', '(no text)')[:60]
        print(f'  [{r.route_decision}] conf={r.confidence:.3f} tokens={r.token_count}  {text!r}')
else:
    print('No records with could_have_been_local=True found in the current dataset.')
    print('(Synthetic records do not set this field — run GristMill in production to populate it.)')

In [ ]:
# Estimate token savings: LLM_NEEDED records that could have been LOCAL_ML
# Assume ~500 tokens average LLM call cost vs 0 for local ML
AVG_LLM_TOKENS = 500
COST_PER_1K_TOKENS = 0.003  # USD, approximate

llm_records = [r for r in all_records if r.route_decision == 'LLM_NEEDED']
saveable = [r for r in llm_records if r.could_have_been_local is True]

total_tokens_saved = len(saveable) * AVG_LLM_TOKENS
cost_saved_usd = total_tokens_saved / 1000 * COST_PER_1K_TOKENS

print('Token savings estimate:')
print(f'  LLM_NEEDED records total  : {len(llm_records)}')
print(f'  Saveable to local ML      : {len(saveable)}')
print(f'  Est. tokens saved         : {total_tokens_saved:,}')
print(f'  Est. cost saved           : ${cost_saved_usd:.4f} USD')
print()
print(f'  Autonomy rate (non-LLM)   : {(len(all_records) - len(llm_records)) / max(len(all_records), 1) * 100:.1f}%')

In [ ]:
from collections import defaultdict
from datetime import timedelta

# Bucket records by day
by_day = defaultdict(lambda: defaultdict(int))
for r in all_records:
    day = datetime.fromtimestamp(r.timestamp_ms / 1000, tz=timezone.utc).date().isoformat()
    by_day[day][r.route_decision] += 1

days = sorted(by_day.keys())
print(f'Routing decisions over {len(days)} distinct days.')

try:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    from datetime import date as date_type

    fig, ax = plt.subplots(figsize=(14, 5))
    route_classes = list(ROUTE_LABEL_MAP.keys())
    colors = ['#4CAF50', '#2196F3', '#FF9800', '#F44336']
    bottom = np.zeros(len(days))

    for cls, color in zip(route_classes, colors):
        vals = np.array([by_day[d].get(cls, 0) for d in days], dtype=float)
        ax.bar(days, vals, bottom=bottom, label=cls, color=color, alpha=0.85)
        bottom += vals

    ax.set_title('Routing Decisions Over Time (stacked bar)', fontsize=13)
    ax.set_xlabel('Date')
    ax.set_ylabel('Record count')
    ax.legend(loc='upper left')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('(matplotlib not installed — skipping time series plot)')
    for day in days[:10]:
        print(f'  {day}: {dict(by_day[day])}')